# 01 - EDA Completa dos Dados de Knowledge Tracing

Este notebook consolida a análise exploratória completa dos dados do projeto MedQ/KT em um único fluxo.

## Objetivos
- validar a qualidade dos dados raw
- entender a distribuição de questões, respostas e temas
- analisar comportamento temporal dos alunos
- inspecionar granularidade por skill e por H2
- gerar evidências para DKT e LPKT

## Fontes esperadas
- `data/raw/questions.json`
- `data/raw/answers.json`
- `data/raw/tree.json`
- `data/processed/mappings/skill_to_h2.json`


In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
SRC_PATH = PROJECT_ROOT / 'src'
if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

from brain_kt.utils.eda_helpers import detect_nested_columns, safe_duplicated_count, safe_log1p

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 140)

RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'

QUESTIONS_PATH = RAW_DIR / 'questions.json'
ANSWERS_PATH = RAW_DIR / 'answers.json'
TREE_PATH = RAW_DIR / 'tree.json'
SKILL_TO_H2_PATH = PROCESSED_DIR / 'mappings' / 'skill_to_h2.json'

print('PROJECT_ROOT:', PROJECT_ROOT)
print('QUESTIONS_PATH exists:', QUESTIONS_PATH.exists())
print('ANSWERS_PATH exists:', ANSWERS_PATH.exists())
print('TREE_PATH exists:', TREE_PATH.exists())
print('SKILL_TO_H2_PATH exists:', SKILL_TO_H2_PATH.exists())

In [ ]:
questions = pd.read_json(QUESTIONS_PATH, encoding='latin-1')
answers = pd.read_json(ANSWERS_PATH)

with open(TREE_PATH, 'r', encoding='utf-8') as f:
    tree = json.load(f)

skill_to_h2 = {}
if SKILL_TO_H2_PATH.exists():
    with open(SKILL_TO_H2_PATH, 'r', encoding='utf-8') as f:
        skill_to_h2 = json.load(f)

print('questions:', len(questions))
print('answers:', len(answers))
print('skill_to_h2 entries:', len(skill_to_h2))

## 1. Amostras e schema

Inspeção inicial para validar colunas, tipos e formato dos objetos.

In [ ]:
display(questions.head(3))
display(answers.head(3))
questions.info()
answers.info()

In [ ]:
print('Nested columns in questions:', detect_nested_columns(questions))
print('Nested columns in answers:', detect_nested_columns(answers))

## 2. Resumo executivo

Métricas centrais para entender o tamanho e a cobertura do dataset.

In [ ]:
summary = pd.Series({
    'n_questions': len(questions),
    'n_answers': len(answers),
    'n_unique_users': answers['user_id'].nunique(),
    'n_unique_question_ids_in_answers': answers['question_id'].nunique(),
    'n_unique_skills_in_answers': answers['skill_id'].nunique(),
    'n_unique_subjects_in_questions': questions['subjectId'].nunique() if 'subjectId' in questions.columns else None,
    'n_unique_institutions': questions['institutionName'].nunique() if 'institutionName' in questions.columns else None,
    'n_unique_exam_years': questions['examYear'].nunique() if 'examYear' in questions.columns else None,
})
summary

## 3. Qualidade dos dados

Análises de nulls, duplicidade e valores inválidos com tratamento seguro para colunas aninhadas.

In [ ]:
nulls_questions = questions.isna().sum().sort_values(ascending=False)
nulls_answers = answers.isna().sum().sort_values(ascending=False)
display(nulls_questions[nulls_questions > 0])
display(nulls_answers[nulls_answers > 0])

In [ ]:
duplicated_questions_rows = safe_duplicated_count(
    questions,
    subset=[c for c in ['questionId', 'subjectId', 'institutionName', 'examYear'] if c in questions.columns],
)

quality_report = pd.Series({
    'duplicated_answers_rows': int(answers.duplicated().sum()),
    'duplicated_questions_rows': duplicated_questions_rows,
    'invalid_correct_values': int((~answers['correct'].isin([0, 1])).sum()),
    'negative_time_response': int((answers['time_response'] < 0).sum()),
    'negative_timestamp': int((answers['timestamp'] < 0).sum()),
})
quality_report

## 4. Distribuição de acertos

Essa etapa ajuda a entender balanceamento do target, algo importante para interpretação de Accuracy e AUC.

In [ ]:
correct_dist = answers['correct'].value_counts().sort_index()
correct_ratio = answers['correct'].value_counts(normalize=True).sort_index()
display(correct_dist)
display(correct_ratio)

In [ ]:
plt.figure(figsize=(6, 4))
correct_dist.plot(kind='bar')
plt.title('Distribuição de respostas corretas/incorretas')
plt.xlabel('correct')
plt.ylabel('Contagem')
plt.xticks(rotation=0)
plt.show()

## 5. Interações por usuário

Knowledge Tracing depende fortemente de sequência. Por isso, é importante medir o tamanho das trajetórias dos alunos.

In [ ]:
user_counts = answers.groupby('user_id').size().sort_values(ascending=False)
user_counts.describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99])

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(user_counts, bins=50)
plt.title('Histograma de interações por usuário')
plt.xlabel('Número de interações')
plt.ylabel('Frequência')
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(user_counts[user_counts > 0], bins=50, log=True)
plt.title('Histograma de interações por usuário (escala log no eixo Y)')
plt.xlabel('Número de interações')
plt.ylabel('Frequência (log)')
plt.show()

In [ ]:
thresholds = [1, 2, 5, 10, 20, 50, 100]
coverage = {f'>={t} interações': int((user_counts >= t).sum()) for t in thresholds}
pd.Series(coverage)

## 6. Cobertura por questão e skill

Aqui avaliamos concentração de respostas em poucas questões ou skills.

In [ ]:
question_counts = answers.groupby('question_id').size().sort_values(ascending=False)
skill_counts = answers.groupby('skill_id').size().sort_values(ascending=False)

display(question_counts.describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]))
display(skill_counts.describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]))

In [ ]:
top_skills = skill_counts.head(20)
plt.figure(figsize=(12, 6))
top_skills.plot(kind='bar')
plt.title('Top 20 skills por número de respostas')
plt.xlabel('skill_id')
plt.ylabel('Contagem de respostas')
plt.xticks(rotation=75)
plt.show()

In [ ]:
top_questions = question_counts.head(20)
plt.figure(figsize=(12, 6))
top_questions.plot(kind='bar')
plt.title('Top 20 questões por número de respostas')
plt.xlabel('question_id')
plt.ylabel('Contagem de respostas')
plt.xticks(rotation=75)
plt.show()

## 7. Sparsity do problema

Uma medida importante em KT é a densidade observada de interações no espaço usuário-skill.

In [ ]:
n_users = answers['user_id'].nunique()
n_skills = answers['skill_id'].nunique()
density = len(answers) / (n_users * n_skills)
pd.Series({
    'n_users': n_users,
    'n_skills': n_skills,
    'n_answers': len(answers),
    'user_skill_density': density,
})

## 8. Tempo de resposta

Tempo de resposta é uma feature importante especialmente para LPKT. Aqui verificamos escala, caudas e necessidade de transformação logarítmica.

In [ ]:
answers['time_response'].describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99])

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(answers['time_response'], bins=50)
plt.title('Distribuição de time_response')
plt.xlabel('time_response')
plt.ylabel('Frequência')
plt.show()

In [ ]:
time_log = safe_log1p(answers['time_response'])
plt.figure(figsize=(10, 5))
plt.hist(time_log, bins=50)
plt.title('Distribuição de log1p(time_response)')
plt.xlabel('log1p(time_response)')
plt.ylabel('Frequência')
plt.show()

## 9. Recência / Delta de tempo

A recência entre interações é um dos sinais mais importantes para modelos temporais.

In [ ]:
answers_sorted = answers.sort_values(['user_id', 'timestamp']).copy()
answers_sorted['delta_t'] = answers_sorted.groupby('user_id')['timestamp'].diff().fillna(0)
answers_sorted['delta_t'].describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99])

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(answers_sorted['delta_t'], bins=50)
plt.title('Distribuição de delta_t')
plt.xlabel('delta_t')
plt.ylabel('Frequência')
plt.show()

In [ ]:
delta_log = safe_log1p(answers_sorted['delta_t'])
plt.figure(figsize=(10, 5))
plt.hist(delta_log, bins=50)
plt.title('Distribuição de log1p(delta_t)')
plt.xlabel('log1p(delta_t)')
plt.ylabel('Frequência')
plt.show()

## 10. Análise das questões

Aqui avaliamos instituições, anos e profundidade temática.

In [ ]:
if 'institutionName' in questions.columns:
    display(questions['institutionName'].value_counts().head(20))

if 'examYear' in questions.columns:
    display(questions['examYear'].value_counts().sort_index())

In [ ]:
if 'examYear' in questions.columns:
    plt.figure(figsize=(10, 4))
    questions['examYear'].value_counts().sort_index().plot(kind='bar')
    plt.title('Distribuição de questões por ano de prova')
    plt.xlabel('Ano')
    plt.ylabel('Número de questões')
    plt.show()

In [ ]:
if 'subjectAccumulatedNames' in questions.columns:
    questions['path_depth'] = questions['subjectAccumulatedNames'].apply(lambda x: len(x) if isinstance(x, list) else 0)
    display(questions['path_depth'].value_counts().sort_index())

    plt.figure(figsize=(8, 4))
    questions['path_depth'].value_counts().sort_index().plot(kind='bar')
    plt.title('Profundidade de caminho temático das questões')
    plt.xlabel('Profundidade')
    plt.ylabel('Número de questões')
    plt.show()

## 11. Enriquecimento com H2

Mapeamos skills para H2 para permitir análise pedagógica agregada.

In [ ]:
answers_h2 = answers.copy()
if skill_to_h2:
    answers_h2['h2'] = answers_h2['skill_id'].astype(str).map(skill_to_h2)
    h2_coverage = answers_h2['h2'].notna().mean()
    print('H2 coverage on answers:', round(h2_coverage, 4))
    display(answers_h2['h2'].value_counts(dropna=False).head(20))
else:
    print('skill_to_h2.json não encontrado.')

In [ ]:
if skill_to_h2:
    top_h2 = answers_h2['h2'].value_counts().head(20)
    plt.figure(figsize=(12, 6))
    top_h2.plot(kind='bar')
    plt.title('Top 20 H2 por número de respostas')
    plt.xlabel('H2')
    plt.ylabel('Contagem de respostas')
    plt.xticks(rotation=75)
    plt.show()

In [ ]:
if skill_to_h2:
    h2_acc = answers_h2.dropna(subset=['h2']).groupby('h2')['correct'].agg(['mean', 'count']).sort_values('count', ascending=False)
    display(h2_acc.head(20))

    top_h2_acc = h2_acc.head(15).sort_values('mean', ascending=False)
    plt.figure(figsize=(12, 6))
    plt.bar(top_h2_acc.index, top_h2_acc['mean'])
    plt.title('Acurácia média por H2 (top 15 por volume)')
    plt.xlabel('H2')
    plt.ylabel('Taxa média de acerto')
    plt.xticks(rotation=75)
    plt.ylim(0, 1)
    plt.show()

## 12. Acurácia por skill

Esse recorte ajuda a entender heterogeneidade de dificuldade entre skills.

In [ ]:
skill_acc = answers.groupby('skill_id')['correct'].agg(['mean', 'count']).sort_values('count', ascending=False)
display(skill_acc.head(20))

In [ ]:
skill_acc_top = skill_acc.head(20).sort_values('mean', ascending=False)
plt.figure(figsize=(12, 6))
plt.bar(skill_acc_top.index.astype(str), skill_acc_top['mean'])
plt.title('Taxa média de acerto por skill (top 20 por volume)')
plt.xlabel('skill_id')
plt.ylabel('Taxa média de acerto')
plt.xticks(rotation=75)
plt.ylim(0, 1)
plt.show()

## 13. Série temporal agregada

Aqui observamos a massa de interações ao longo do tempo.

In [ ]:
answers_time = answers.copy()
answers_time['timestamp_dt'] = pd.to_datetime(answers_time['timestamp'], unit='ms', errors='coerce')
answers_time['date'] = answers_time['timestamp_dt'].dt.date
daily_counts = answers_time.groupby('date').size()
daily_counts.describe()

In [ ]:
plt.figure(figsize=(12, 5))
daily_counts.plot()
plt.title('Interações diárias ao longo do tempo')
plt.xlabel('Data')
plt.ylabel('Número de interações')
plt.show()

## 14. Merge entre respostas e questões

Serve para validar consistência entre question_id e metadados das questões.

In [ ]:
question_cols = [c for c in ['questionId', 'subjectId', 'institutionName', 'examYear', 'subjectAccumulatedNames'] if c in questions.columns]
merged = answers.merge(questions[question_cols], left_on='question_id', right_on='questionId', how='left')
merged_match_rate = merged['questionId'].notna().mean()
pd.Series({
    'merge_match_rate': merged_match_rate,
    'missing_question_metadata_rows': int(merged['questionId'].isna().sum()),
})

## 15. Principais achados

Preencher após execução final:

- balanceamento do target
- dispersão de interações por usuário
- caudas longas em tempo de resposta e recência
- concentração de respostas em poucas skills/H2
- cobertura do mapeamento skill -> H2
- implicações para DKT e LPKT

## Leituras esperadas
- `time_response` e `delta_t` devem justificar transformação logarítmica
- distribuição por usuário deve justificar filtragem mínima de interações
- análise por skill e H2 deve apoiar a decisão entre granularidade item-level e agregação temática
